# Bottom Detection Verification and Failure Analysis

Visual verification toolkit for bottom detection methods. Where notebook 21
asks which method is best overall, this notebook asks where and why methods
fail — disagreement maps, failure classification, frequency comparison,
and diagnostic galleries of the hardest cases.

In [ ]:
import sys
sys.path.insert(0, '../../..')
from project_paths import REPO_ROOT


import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from raw_sensor_integration.data_loader import load_zarr
from raw_sensor_integration.bottom_labelling import (
    detect_bottom_tvg,
    detect_bottom_dp,
    detect_bottom_gradient,
    detect_bottom_matched_filter,
    detect_bottom_temporal,
    detect_bottom_morphological,
    detect_bottom_cfar,
    detect_bottom_wavelet,
    detect_bottom_changepoint
)

plt.rcParams['figure.dpi'] = 120
plt.rcParams['figure.facecolor'] = 'white'

ZARR_DIR = REPO_ROOT / 'Datasets/echopype_zarr'
FREQ_PRIMARY = 38000
FREQ_SECONDARY = 200000

METHODS = {
    'tvg':          lambda p, r: detect_bottom_tvg(p, r, minimum_range=5.0),
    'dp':           lambda p, r: detect_bottom_dp(p, r, smoothness=1.0, minimum_range=5.0),
    'gradient':     lambda p, r: detect_bottom_gradient(p, r, minimum_range=5.0),
    'matched_filter': lambda p, r: detect_bottom_matched_filter(p, r, minimum_range=5.0),
    'temporal':     lambda p, r: detect_bottom_temporal(p, r, minimum_range=5.0),
    'morphological': lambda p, r: detect_bottom_morphological(p, r, minimum_range=5.0),
    'cfar':         lambda p, r: detect_bottom_cfar(p, r, minimum_range=5.0),
    'wavelet':      lambda p, r: detect_bottom_wavelet(p, r, minimum_range=5.0),
    'changepoint':  lambda p, r: detect_bottom_changepoint(p, r, minimum_range=5.0),
}

METHOD_COLORS = {
    'tvg':           '#e41a1c',
    'dp':            '#377eb8',
    'gradient':      '#4daf4a',
    'matched_filter':'#984ea3',
    'temporal':      '#ff7f00',
    'morphological': '#a65628',
    'cfar':          '#f781bf',
    'wavelet':       '#ffff33',
    'changepoint':   '#00ced1',
}

## Load Data and Run All Detectors

Using a larger subset (20 files) for more comprehensive failure analysis.

In [ ]:
all_zarrs = sorted(ZARR_DIR.glob('*.zarr'))
n_files = 20
indices = np.linspace(0, len(all_zarrs) - 1, n_files, dtype=int)
selected_zarrs = [all_zarrs[i] for i in indices]

results = {}
file_data = {}

for zarr_path in selected_zarrs:
    fname = zarr_path.name
    print(f'Processing {fname}...', end=' ')

    data = load_zarr(zarr_path, frequency=FREQ_PRIMARY)
    power_db = data.sv_db
    range_array = data.range_array

    file_data[fname] = (power_db, range_array)
    results[fname] = {}

    for method_name, method_fn in METHODS.items():
        try:
            results[fname][method_name] = method_fn(power_db, range_array)
        except Exception:
            results[fname][method_name] = np.full(power_db.shape[1], np.nan)

    print('done')

print(f'\nLoaded {len(results)} files.')

## Disagreement Maps

For each file, a heatmap showing how many methods place the bottom
near each (sample, ping) position. Bright regions indicate strong
agreement; dark regions indicate disagreement or no detections.

In [ ]:
AGREEMENT_RADIUS = 3  # samples

for fname in list(results.keys())[:5]:  # show first 5 files
    power_db, range_array = file_data[fname]
    n_samples, n_pings = power_db.shape
    sample_dist = float(range_array[1] - range_array[0])

    agreement_img = np.zeros((n_samples, n_pings), dtype=np.float32)

    for method_name in METHODS:
        bottom = results[fname][method_name]
        for p in range(n_pings):
            if np.isnan(bottom[p]):
                continue
            row = int(round((bottom[p] - range_array[0]) / sample_dist))
            row = np.clip(row, 0, n_samples - 1)
            r_lo = max(0, row - AGREEMENT_RADIUS)
            r_hi = min(n_samples, row + AGREEMENT_RADIUS + 1)
            agreement_img[r_lo:r_hi, p] += 1

    agreement_img /= len(METHODS)

    vmin_echo, vmax_echo = np.nanpercentile(power_db, [2, 98])

    fig, axes = plt.subplots(1, 2, figsize=(16, 5), sharex=True, sharey=True)

    axes[0].imshow(power_db, aspect='auto', cmap='viridis',
                   vmin=vmin_echo, vmax=vmax_echo, origin='upper',
                   extent=[0, n_pings, range_array[-1], range_array[0]])
    axes[0].set_title(f'{fname} — echogram')
    axes[0].set_ylabel('Range (m)')

    im = axes[1].imshow(agreement_img, aspect='auto', cmap='hot',
                        vmin=0, vmax=1, origin='upper',
                        extent=[0, n_pings, range_array[-1], range_array[0]])
    axes[1].set_title('Method agreement (fraction)')
    plt.colorbar(im, ax=axes[1], label='Agreement fraction')

    for ax in axes:
        ax.set_xlabel('Ping')

    plt.tight_layout()
    plt.show()

## Failure Classification

Categorise each method's failures into four types:
- **Gap**: returns NaN while the consensus has a valid detection
- **Outlier**: detection is >5 m from consensus
- **Shallow bias**: consistently shallower than consensus (median delta < -1 m)
- **Deep bias**: consistently deeper than consensus (median delta > 1 m)

In [ ]:
OUTLIER_THRESH = 5.0  # meters
BIAS_THRESH = 1.0     # meters

failure_counts = {m: {'gap': 0, 'outlier': 0, 'shallow_bias': 0, 'deep_bias': 0, 'total_pings': 0}
                  for m in METHODS}

for fname in results:
    all_traces = np.array([results[fname][m] for m in METHODS])
    consensus = np.nanmedian(all_traces, axis=0)

    for method_name in METHODS:
        bottom = results[fname][method_name]
        fc = failure_counts[method_name]

        for p in range(len(consensus)):
            if np.isnan(consensus[p]):
                continue
            fc['total_pings'] += 1

            if np.isnan(bottom[p]):
                fc['gap'] += 1
            elif abs(bottom[p] - consensus[p]) > OUTLIER_THRESH:
                fc['outlier'] += 1

    # Bias: computed per file over valid pings
    for method_name in METHODS:
        bottom = results[fname][method_name]
        both = ~np.isnan(bottom) & ~np.isnan(consensus)
        if both.sum() > 10:
            delta = np.median(bottom[both] - consensus[both])
            if delta < -BIAS_THRESH:
                failure_counts[method_name]['shallow_bias'] += 1
            elif delta > BIAS_THRESH:
                failure_counts[method_name]['deep_bias'] += 1

# Plot stacked bar chart
method_list = list(METHODS.keys())
failure_types = ['gap', 'outlier', 'shallow_bias', 'deep_bias']
colors = ['#636363', '#e41a1c', '#377eb8', '#ff7f00']

fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(len(method_list))
bottoms = np.zeros(len(method_list))

for ft, color in zip(failure_types, colors):
    vals = [failure_counts[m][ft] for m in method_list]
    ax.bar(x, vals, bottom=bottoms, label=ft, color=color, alpha=0.8)
    bottoms += vals

ax.set_xticks(x)
ax.set_xticklabels(method_list, rotation=45, ha='right')
ax.set_ylabel('Count')
ax.set_title('Failure classification by method')
ax.legend()
plt.tight_layout()
plt.show()

# Print table
fail_df = pd.DataFrame(failure_counts).T
fail_df['gap_rate'] = (fail_df['gap'] / fail_df['total_pings']).round(3)
fail_df['outlier_rate'] = (fail_df['outlier'] / fail_df['total_pings']).round(3)
print(fail_df[['gap', 'outlier', 'shallow_bias', 'deep_bias', 'gap_rate', 'outlier_rate']].to_string())

## Frequency Comparison

Run each detector on both 38 kHz and 200 kHz independently. Scatter
plot of 38 kHz vs 200 kHz bottom depth — points on the diagonal
indicate agreement; off-diagonal reveals frequency-dependent failures.

In [ ]:
# Pick 5 files for cross-frequency analysis
freq_zarrs = selected_zarrs[:5]

freq_results = {}  # {method: {freq: [all bottom values]}}
for m in METHODS:
    freq_results[m] = {'38k': [], '200k': []}

for zarr_path in freq_zarrs:
    fname = zarr_path.name
    print(f'Cross-frequency: {fname}')

    try:
        data_38 = load_zarr(zarr_path, frequency=FREQ_PRIMARY)
        data_200 = load_zarr(zarr_path, frequency=FREQ_SECONDARY)
    except Exception as e:
        print(f'  Skipping — {e}')
        continue

    # Align to same number of pings
    n_pings = min(data_38.sv_db.shape[1], data_200.sv_db.shape[1])

    for method_name, method_fn in METHODS.items():
        try:
            b38 = method_fn(data_38.sv_db[:, :n_pings], data_38.range_array)
            b200 = method_fn(data_200.sv_db[:, :n_pings], data_200.range_array)
        except Exception:
            continue

        both = ~np.isnan(b38) & ~np.isnan(b200)
        freq_results[method_name]['38k'].extend(b38[both].tolist())
        freq_results[method_name]['200k'].extend(b200[both].tolist())

# Scatter plots
fig, axes = plt.subplots(3, 3, figsize=(14, 12))
axes = axes.flatten()

for idx, method_name in enumerate(METHODS):
    ax = axes[idx]
    x = np.array(freq_results[method_name]['38k'])
    y = np.array(freq_results[method_name]['200k'])

    if len(x) > 0:
        ax.scatter(x, y, s=1, alpha=0.3, color=METHOD_COLORS[method_name])
        lims = [min(x.min(), y.min()), max(x.max(), y.max())]
        ax.plot(lims, lims, 'k--', linewidth=0.5, alpha=0.5)
        mad = np.median(np.abs(x - y))
        ax.text(0.05, 0.95, f'MAD: {mad:.1f} m', transform=ax.transAxes,
                fontsize=8, va='top')

    ax.set_title(method_name, fontsize=9)
    ax.set_xlabel('38 kHz (m)', fontsize=7)
    ax.set_ylabel('200 kHz (m)', fontsize=7)
    ax.tick_params(labelsize=7)

plt.suptitle('38 kHz vs 200 kHz bottom detection', fontsize=12)
plt.tight_layout()
plt.show()

## Gallery of Hardest Pings

Column-level visualisation of the 20 pings where methods disagree
the most. Each panel shows the power profile along depth with
horizontal markers for each method's detection.

In [ ]:
# Collect per-ping disagreement across all files
ping_records = []  # (fname, ping_idx, std)

for fname in results:
    all_traces = np.array([results[fname][m] for m in METHODS])
    with np.errstate(all='ignore'):
        stds = np.nanstd(all_traces, axis=0)
    for p in range(len(stds)):
        if not np.isnan(stds[p]) and stds[p] > 0:
            ping_records.append((fname, p, stds[p]))

ping_records.sort(key=lambda x: -x[2])
top_hard = ping_records[:20]

fig, axes = plt.subplots(4, 5, figsize=(18, 14))
axes = axes.flatten()

for idx, (fname, ping_idx, std_val) in enumerate(top_hard):
    ax = axes[idx]
    power_db, range_array = file_data[fname]
    col = power_db[:, ping_idx]

    ax.plot(col, range_array, 'k-', linewidth=0.5, alpha=0.7)

    for method_name in METHODS:
        b = results[fname][method_name][ping_idx]
        if not np.isnan(b):
            # Find the dB value at the detected depth
            sample_dist = float(range_array[1] - range_array[0])
            row = int(round((b - range_array[0]) / sample_dist))
            row = np.clip(row, 0, len(col) - 1)
            ax.plot(col[row], b, 'o', color=METHOD_COLORS[method_name],
                    markersize=4, label=method_name)

    ax.set_title(f'p{ping_idx} std={std_val:.1f}m', fontsize=7)
    ax.tick_params(labelsize=6)
    ax.invert_yaxis()

    if idx == 0:
        ax.legend(fontsize=5, loc='lower left')

plt.suptitle('20 hardest pings (highest cross-method std)', fontsize=11)
plt.tight_layout()
plt.show()

## Gallery of Hardest Files

Files ranked by average method disagreement. The top 5 hardest files
shown with full echogram overlays.

In [ ]:
file_difficulty = {}
for fname in results:
    all_traces = np.array([results[fname][m] for m in METHODS])
    with np.errstate(all='ignore'):
        stds = np.nanstd(all_traces, axis=0)
    file_difficulty[fname] = np.nanmean(stds)

hardest_files = sorted(file_difficulty.items(), key=lambda x: -x[1])[:5]

for fname, difficulty in hardest_files:
    power_db, range_array = file_data[fname]
    n_samples, n_pings = power_db.shape
    vmin, vmax = np.nanpercentile(power_db, [2, 98])

    fig, ax = plt.subplots(figsize=(16, 6))
    ax.imshow(power_db, aspect='auto', cmap='viridis',
              vmin=vmin, vmax=vmax, origin='upper',
              extent=[0, n_pings, range_array[-1], range_array[0]])

    for method_name in METHODS:
        bottom = results[fname][method_name]
        valid = ~np.isnan(bottom)
        if valid.sum() == 0:
            continue
        pings = np.arange(n_pings)
        ax.plot(pings[valid], bottom[valid],
                color=METHOD_COLORS[method_name],
                linewidth=1.0, alpha=0.8, label=method_name)

    ax.set_xlabel('Ping')
    ax.set_ylabel('Range (m)')
    ax.set_title(f'{fname} — avg disagreement {difficulty:.1f} m')
    ax.legend(loc='upper right', fontsize=7, ncol=3)
    plt.tight_layout()
    plt.show()

## New Method Diagnostics

For CFAR, wavelet, and changepoint: show the intermediate computation
at a few hard pings to understand what each method "sees" when it fails.
- CFAR: adaptive threshold curve vs signal
- Wavelet: CWT scalogram
- Changepoint: CUSUM statistic along depth

In [ ]:
import scipy.signal as sig
from raw_sensor_integration.bottom_labelling.dp import _apply_pseudo_tvg
from raw_sensor_integration.bottom_labelling.changepoint import _cusum_statistic

# Pick 3 hard pings for diagnostics
diag_pings = top_hard[:3]

for fname, ping_idx, std_val in diag_pings:
    power_db, range_array = file_data[fname]
    n_samples = power_db.shape[0]
    sample_dist = float(range_array[1] - range_array[0])
    min_index = max(0, int(5.0 / sample_dist))

    col_db = power_db[:, ping_idx]

    # TVG-corrected column
    corrected = _apply_pseudo_tvg(power_db[:, ping_idx:ping_idx+1], range_array)
    col_tvg = corrected[:, 0]

    fig, axes = plt.subplots(1, 4, figsize=(20, 5))
    fig.suptitle(f'{fname} ping {ping_idx} (std={std_val:.1f} m)', fontsize=10)

    # Panel 1: Raw power profile
    axes[0].plot(col_db, range_array, 'k-', linewidth=0.7)
    axes[0].set_xlabel('Power (dB)')
    axes[0].set_ylabel('Range (m)')
    axes[0].set_title('Power profile')
    axes[0].invert_yaxis()

    # Panel 2: CFAR — threshold vs signal
    col_linear = 10.0 ** (col_tvg / 10.0)
    n_guard, n_train = 5, 20
    pfa = 1e-3
    alpha = n_train * (pfa ** (-1.0 / n_train) - 1.0)

    # Compute CFAR threshold at each sample
    half_train = n_train // 2
    threshold_curve = np.full(n_samples, np.nan)
    for i in range(n_guard + half_train, n_samples - n_guard - half_train):
        lead = col_linear[i - n_guard - half_train:i - n_guard]
        lag_end = min(i + n_guard + 1 + half_train, n_samples)
        lag = col_linear[i + n_guard + 1:lag_end]
        noise = np.mean(np.concatenate([lead, lag]))
        threshold_curve[i] = alpha * noise

    axes[1].plot(col_linear, range_array, 'k-', linewidth=0.7, label='Signal')
    valid_t = ~np.isnan(threshold_curve)
    axes[1].plot(threshold_curve[valid_t], range_array[valid_t],
                 'r-', linewidth=0.7, alpha=0.8, label='CFAR threshold')
    axes[1].set_xlabel('Linear power')
    axes[1].set_title('CFAR')
    axes[1].legend(fontsize=7)
    axes[1].invert_yaxis()

    # Panel 3: Wavelet CWT scalogram
    vmin_n = np.nanpercentile(col_tvg[min_index:], 1)
    vmax_n = np.nanpercentile(col_tvg[min_index:], 99)
    col_norm = np.clip((col_tvg - vmin_n) / max(vmax_n - vmin_n, 1e-6), 0, 1)
    scales = np.geomspace(2.0, 30.0, 12)
    cwt_matrix = sig.cwt(col_norm, sig.ricker, scales)

    axes[2].imshow(np.abs(cwt_matrix), aspect='auto', cmap='hot',
                   extent=[range_array[0], range_array[-1], scales[-1], scales[0]])
    axes[2].set_xlabel('Range (m)')
    axes[2].set_ylabel('Scale')
    axes[2].set_title('Wavelet scalogram')

    # Panel 4: CUSUM statistic
    cusum = _cusum_statistic(col_norm, min_index)
    axes[3].plot(cusum, range_array, 'b-', linewidth=0.7)
    axes[3].set_xlabel('|CUSUM|')
    axes[3].set_title('Change-point (CUSUM)')
    axes[3].invert_yaxis()

    plt.tight_layout()
    plt.show()

## Depth-Dependent Bias

Detection error (method minus consensus) as a function of depth. A
systematic trend indicates depth-dependent bias in the method.

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(14, 12))
axes = axes.flatten()

for idx, method_name in enumerate(METHODS):
    ax = axes[idx]
    all_depths = []
    all_errors = []

    for fname in results:
        all_traces = np.array([results[fname][m] for m in METHODS])
        consensus = np.nanmedian(all_traces, axis=0)
        bottom = results[fname][method_name]

        both = ~np.isnan(bottom) & ~np.isnan(consensus)
        if both.sum() > 0:
            all_depths.extend(consensus[both].tolist())
            all_errors.extend((bottom[both] - consensus[both]).tolist())

    if len(all_depths) > 0:
        depths = np.array(all_depths)
        errors = np.array(all_errors)

        # Subsample for plotting
        if len(depths) > 5000:
            idx_sub = np.random.default_rng(42).choice(len(depths), 5000, replace=False)
            depths_plot, errors_plot = depths[idx_sub], errors[idx_sub]
        else:
            depths_plot, errors_plot = depths, errors

        ax.scatter(depths_plot, errors_plot, s=1, alpha=0.2,
                   color=METHOD_COLORS[method_name])
        ax.axhline(0, color='black', linewidth=0.5)

        # Binned median trend
        bins = np.linspace(depths.min(), depths.max(), 20)
        bin_idx = np.digitize(depths, bins)
        for b in range(1, len(bins)):
            mask = bin_idx == b
            if mask.sum() > 10:
                ax.plot(bins[b-1], np.median(errors[mask]), 'ks', markersize=3)

    ax.set_title(method_name, fontsize=9)
    ax.set_xlabel('Depth (m)', fontsize=7)
    ax.set_ylabel('Error (m)', fontsize=7)
    ax.set_ylim(-15, 15)
    ax.tick_params(labelsize=7)

plt.suptitle('Detection error vs depth (method minus consensus)', fontsize=11)
plt.tight_layout()
plt.show()